# Noise-immune correlations: adjoint, autodiff, and stochastic approach

The purpose of this example is to demonstrate the consistency of different approaches to computing gradients: adjoint based approaches (highly custom), automatic differentiation (),

The example we choose is Raman soliton propagation in single-mode fiber and sub-Poissonian light (intensity squeezing) induced by spectral filtering. We look at the sensitivity structure of variances to quantum and excess noise and show an example of the type of noise-immune quantum correlations described in https://www.nature.com/articles/s41566-025-01677-2.

In [ ]:
using Pkg

const ROOT = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(ROOT)

using LinearAlgebra
using Printf
using Random
using Statistics
using FFTW
using PyPlot
using DelimitedFiles

isdefined(Main, :PulsePropagation) ||
    include(joinpath(ROOT, "src", "PulsePropagation.jl"))
using .PulsePropagation

const c_light_m_ps = 2.99792458e-4
const c_light_m_s = 2.99792458e8
const hbar = 1.054571817e-34
const kb = 1.380649e-23

In [ ]:
## Grid and simulation properties
Nt::Int = 2^12;
time_window_ps::Float64 = 50.0;
lambda0_m::Float64 = 1550e-9;
f0 = c_light_m_s / lambda0_m * 1e-12; ## THz

grid = TimeGrid(Nt, time_window_ps)

t = time_axis(grid)                         # ps, centered
f = frequency_axis(grid; shifted=true)      # THz relative frequency grid
λ = wavelength_axis(grid, f0; shifted=true) # nm
w = photon_weights(grid, f0; shifted=false) # photon-bin weights

## Fiber properties
L_fiber::Float64 = 10;

beta2_si::Float64 = -2.06e-26;
beta3_si::Float64 = 8.1038e-41;

beta2_ps2_m::Float64 = -2.06e-26 * 1e24;
beta3_ps3_m::Float64 = 8.1038e-41 * 1e36;
betas = reshape([0.0, 0.0, beta2_ps2_m, beta3_ps3_m], :, 1);

n2::Float64 = 2.3e-20;
mfd_um::Float64 = 9.2;
Aeff_m2::Float64 = π * (mfd_um * 1e-6 / 2)^2;
sr = reshape([1 / Aeff_m2], 1, 1, 1, 1);
gamma = 2*π*n2/(lambda0_m*Aeff_m2)


f_raman::Float64 = 0.18;
tau1_raman_fs::Float64 = 12.2;
tau2_raman_fs::Float64 = 32.0;
temperature::Float64 = 300.0;

system = PassiveFiber(; length = L_fiber, lambda0 = lambda0_m, betas = betas, f0 = f0, material=AgarwalRaman(; fraction=f_raman, tau1_fs=tau1_raman_fs, tau2_fs=tau2_raman_fs, temperature_K=temperature), sr = sr, mfd = mfd_um, n2 = n2, raman_fraction = f_raman, source_path = nothing)
dofs   = SingleModeField()
terms = PropagationTerms(Dispersion(), Kerr(), Raman());

## Pulse properties
pulse_fwhm_ps::Float64 = 0.22;
peak_power::Float64 = 3655.7;

fields = sech_pulse(grid; peak_power = peak_power, fwhm=pulse_fwhm_ps,
                    dofs=dofs,
                    transform=("ifft", 0.0),
                    time_offset=0.0);

In [ ]:
figure(figsize=(3,3))
plot(t, abs2.(fields[:,1,1]))
xlim(-1,1)
xlabel("t (ps)")
ylabel("Power (W)")
display(gcf())

In [ ]:
N_z = 501;
zsave = collect(range(0.0, L_fiber; length=N_z))

solver = RK4IPSolver(;
    stepping = AdaptiveStep(
        initial_dz = 1e-5,
        max_dz = 1e-3,
        rtol = 1e-8,
        atol = 1e-8,
    ),
    saveat = SaveAt(zsave),
)

In [ ]:
problem = PulsePropagationProblem(; grid, system, dofs, terms, solver, fields)
sol = solve(problem)
spec_out = spectrum(sol);
spec_z = z_spectrum(sol; shifted=true, sum_modes=false);

In [ ]:
figure(figsize=(3,3))
plot(λ, spectrum(sol))
xlim(1500,1700)
display(gcf())

In [ ]:
# figure(figsize=(3,3))
# contourf(λ, sol.output.z, 10log10.(spec_z[:,1,:]'), levels=[-110:2.5:0;])
# colorbar()
# xlabel("Wavelength (nm)")
# ylabel("z (m)")
# xlim(1400,1700)
# clim(-40,0)
# xlabel("λ (nm)")
# ylabel("z (m)")
# display(gcf())

Peak Raman wavelength

In [ ]:
maxind = argmax(fftshift(spec_out)) 
filter = zero(spec_out);
filter[maxind] = 1.0;
max_filt = filter;

obs = FilterEnergy(; filter=max_filt .* w, domain=:frequency, shifted=false, modes=:all)
photon_obs_peak = SpectralPhotonNumber(; filter=max_filt, modes=:all, shifted=false)

n_out = value(obs, sol);
lambda_L = terminal_condition(obs, sol);

@assert isapprox(n_out, value(photon_obs_peak, sol); rtol=1e-12)
@assert norm(lambda_L .- terminal_condition(photon_obs_peak, sol)) / norm(lambda_L) < 1e-12

In [ ]:
grad_rs = gradient(
    problem,
    obs;
    trajectory=sol,
    method=PulsePropagation.Adjoint(),
    wrt=InitialField(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18,
    dz_adj=1e-3,
    reltol=1e-10,
    abstol=1e-10,
)

In [ ]:
figure(figsize=(3,3))
semilogy(λ, fftshift(abs2.(grad_rs.initial_field_gradient)))
xlim(1400,1700)
ylim(1)
xlabel("λ (nm)")
ylabel("Raman peak sensitivity (a.u)")
display(gcf())

In [ ]:
var = sum(abs2, grad_rs.initial_field_gradient);
fano = var / n_out

variance_peak = VarianceObjective(obs;
    method=PulsePropagation.Adjoint(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18,
    dz_adj=1e-3,
    reltol=1e-10,
    abstol=1e-10,
)
variance_peak_result = variance_result(variance_peak, problem, sol)

@assert isapprox(variance_peak_result.value, n_out; rtol=1e-12)
@assert isapprox(variance_peak_result.variance, var; rtol=1e-10)
@assert isapprox(variance_peak_result.fano, fano; rtol=1e-10)

fano, variance_peak_result.fano

Squeezing filter

In [ ]:
opt_filt_trunc = readdlm(joinpath(@__DIR__, "optimal_filter.csv"));
opt_filt = zeros(Nt);
opt_filt[1196:2371] = reverse(opt_filt_trunc); 

λ_grid_adj = λ[1196:2371]; # hard coded numbers based on example - has to do with computing Jacobian (∂n_i/∂u_j) not over full wavelength band of simulation, which is very expensive and most wavelengths don't participate in dynamics
ww_index_grid = unique([argmin(abs.(fftshift(λ) .- ll)) for ll in λ_grid_adj]);
ww_index_grid = reverse(ww_index_grid); # This gets the weights and fields arranged in wavelengths low to high, same as the filter.

opt_filt_unshifted = fftshift(opt_filt);

In [ ]:
figure(figsize=(3,3))
plot(λ,opt_filt)
xlim(1500,1700)
xlabel("λ (nm)")
ylabel("Filter transmission")
display(gcf())

In [ ]:
obs = FilterEnergy(; filter=opt_filt_unshifted .* w, domain=:frequency, shifted=false, modes=:all)
photon_obs_filter = SpectralPhotonNumber(; filter=opt_filt_unshifted, modes=:all, shifted=false)

n_out = value(obs, sol);
lambda_L = terminal_condition(obs, sol);

@assert isapprox(n_out, value(photon_obs_filter, sol); rtol=1e-12)
@assert norm(lambda_L .- terminal_condition(photon_obs_filter, sol)) / norm(lambda_L) < 1e-12

In [ ]:
n_out

In [ ]:
grad_adj = gradient(
    problem,
    photon_obs_filter;
    trajectory=sol,
    method=PulsePropagation.Adjoint(),
    wrt=InitialField(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18,
    dz_adj=1e-3,
    reltol=1e-10,
    abstol=1e-10,
)

In [ ]:
F0, λ0, Δλ = 10, 1550, 20;
fano_list = F0*exp.(-(λ .- λ0) .^2 ./ (Δλ)^2) .+ 1;

figure(figsize=(3,3))
plot(λ,fano_list)
xlim(1500,1600)
xlabel("λ (nm)")
ylabel("Fano factor")
display(gcf())

In [ ]:
figure(figsize=(3,3))
plot(λ, fftshift(abs2.(grad_adj.initial_field_gradient)))
plot(λ, fftshift(abs2.(grad_adj.initial_field_gradient)) .* fano_list)
xlim(1530,1675)
ylim(1e4)
xlabel("λ (nm)")
ylabel("Filtered output sensitivity (a.u)")
display(gcf())

In [ ]:
var = sum(abs2, grad_adj.initial_field_gradient);
fano = var / n_out

variance_filter = VarianceObjective(obs;
    method=PulsePropagation.Adjoint(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18,
    dz_adj=1e-3,
    reltol=1e-10,
    abstol=1e-10,
)
variance_filter_result = variance_result(variance_filter, problem, sol)

@assert isapprox(variance_filter_result.value, n_out; rtol=1e-12)
@assert isapprox(variance_filter_result.variance, var; rtol=1e-10)
@assert isapprox(variance_filter_result.fano, fano; rtol=1e-10)

fano, variance_filter_result.fano

In [ ]:
fano_noise_input = sum(fano_list .* fftshift(abs2.(grad_adj.initial_field_gradient))) / n_out

Raman contribution

In [ ]:
Cω = PulsePropagation.raman_langevin_spectrum(
    Nt, diff(t)[1], f0, gamma;
    raman_fraction=0.18,
    temperature_K=300.0,
);

In [ ]:
Azt = sol.output.fields;
λzt = (conj(fft(grad_adj.adjoint_trajectory.lambdaw,1)));
Bzt = -2*imag(Azt .* λzt);
Bzω = ifft(Bzt,1);
dn2_raman = gamma^2 * sum(abs2.(Bzω) .* Cω) * (diff(f)[1] * 1e12) * (L_fiber / N_z)

In [ ]:
dn2_raman_frac = dn2_raman / sum(fftshift(abs2.(grad_adj.initial_field_gradient)))
fano_with_raman = (sum(fftshift(abs2.(grad_adj.initial_field_gradient))) + dn2_raman) / n_out

Stochastic check

In [ ]:
N_z_stochastic = 501
ntraj_stochastic = 600

raman_added_mc = stochastic_raman_added_variance(
    problem,
    obs;
    ntraj=ntraj_stochastic,
    dz_stochastic=L_fiber / (N_z_stochastic - 1),
    drive=RamanLangevin(:agarwal;
        fraction=f_raman,
        temperature_K=temperature,
        gamma_eff=gamma,
    ),
    seed_initial=4321,
    seed_raman=5678,
    progress=true,
);


In [ ]:
@printf("adjoint Raman added variance        = %.6e\n", dn2_raman)
@printf("MC paired Raman increment variance = %.6e\n", raman_added_mc.var_paired_increment)
@printf("MC total - initial variance        = %.6e\n", raman_added_mc.var_added_difference)
@printf("adjoint Raman added Fano            = %.6e\n", dn2_raman / n_out)
@printf("MC paired Raman increment Fano     = %.6e\n", raman_added_mc.fano_paired_increment)
@printf("MC total - initial Fano            = %.6e\n", raman_added_mc.fano_added_difference)
@printf("MC initial-only variance            = %.6e\n", raman_added_mc.var_initial)
@printf("MC initial-only Fano                = %.6e\n", raman_added_mc.fano_initial)
@printf("MC total Fano                       = %.6e\n", raman_added_mc.fano_total)

AD check

In [ ]:
solver = RK4IPSolver(;
    stepping = FixedStep(
        dz = 1e-2;
    ),
    saveat = SaveAt(zsave),
)

problem = PulsePropagationProblem(; grid, system, dofs, terms, solver, fields)
sol = solve(problem)

In [ ]:
grad_ad = gradient(
    problem,
    obs;
    trajectory=sol,
    method=AutomaticDifferentiation(),
    wrt=InitialField(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18
)

In [ ]:
var = sum(abs2, grad_ad.initial_field_gradient);
fano = var / n_out

In [ ]:
fano_noise_input = sum(fano_list .* fftshift(abs2.(grad_ad.initial_field_gradient))) / n_out

In [ ]:
figure(figsize=(3,3))
semilogy(λ, fftshift(abs2.(grad_adj.initial_field_gradient)))
semilogy(λ, fftshift(abs2.(grad_ad.initial_field_gradient)),"--")
xlim(1530,1675)
ylim(1e4)
xlabel("λ (nm)")
ylabel("Filtered output sensitivity (a.u)")
display(gcf())

In [ ]:
sum(abs2.(grad_adj.initial_field_gradient))

In [ ]:
sum(abs2.(grad_ad.initial_field_gradient))

In [ ]:
figure(figsize=(2,1))
plot(t, abs2.(fields[:,1,1]))
xlim(-1,1)
xlabel("t (ps)")
ylabel("Power (W)")
display(gcf())

In [ ]:
figure(figsize=(2,3))
pcolormesh(λ, sol.output.z, 10log10.(spec_z[:,1,:]'))
colorbar()
xlabel("Wavelength (nm)")
ylabel("z (m)")
xlim(1400,1700)
clim(-40,0)
xlabel("λ (nm)")
ylabel("z (m)")
display(gcf())

In [ ]:
figure(figsize=(2,2))
semilogy(λ, fftshift(abs2.(grad_rs.initial_field_gradient)))
xlim(1400,1700)
ylim(1)
xlabel("λ (nm)")
ylabel("Raman peak \n sensitivity (a.u)")
display(gcf())

In [ ]:
figure(figsize=(2,2))
semilogy(λ, fftshift(abs2.(grad_adj.initial_field_gradient)))
semilogy(λ, fftshift(abs2.(grad_ad.initial_field_gradient)),"--")
xlim(1530,1675)
ylim(1e4)
xlabel("λ (nm)")
ylabel("Filtered output \n sensitivity (a.u)")
display(gcf())

In [ ]:
figure(figsize=(5,1))
plot(λ,opt_filt)
xlim(1550,1680)
xlabel("λ (nm)")
ylabel("Transmission")
display(gcf())

<!-- opt_filt_trunc = readdlm(joinpath(@__DIR__, "optimal_filter.csv"));
opt_filt = zeros(p.Nt);
opt_filt[1196:2371] = reverse(opt_filt_trunc); 

λ_grid_adj = λ[1196:2371]; # hard coded numbers based on example - has to do with computing Jacobian (∂n_i/∂u_j) not over full wavelength band of simulation, which is very expensive and most wavelengths don't participate in dynamics
ww_index_grid = unique([argmin(abs.(fftshift(λ) .- ll)) for ll in λ_grid_adj]);
ww_index_grid = reverse(ww_index_grid); -->

In [ ]:
figure(figsize=(10,8))

ax_power = subplot2grid((5,9), (0,0), rowspan=1, colspan=4)
ax_spec = subplot2grid((5,9), (1,0), rowspan=3, colspan=4)
ax_raman = subplot2grid((5,9), (0,5), rowspan=2, colspan=4)
ax_filtered = subplot2grid((5,9), (2,5), rowspan=2, colspan=4)
ax_filter = subplot2grid((5,9), (4,0), rowspan=1, colspan=9)

ax_power.plot(t, abs2.(fields[:,1,1]))
ax_power.set_xlim(-1, 1)
ax_power.set_xlabel("t (ps)")
ax_power.set_ylabel("Power (W)")

spec_plot = ax_spec.pcolormesh(λ, sol.output.z, 10log10.(spec_z[:,1,:]'))
spec_plot.set_clim(-40, 0)
colorbar(spec_plot, ax=ax_spec)
ax_spec.set_xlim(1400, 1700)
ax_spec.set_xlabel("λ (nm)")
ax_spec.set_ylabel("z (m)")

raman_line, = ax_raman.semilogy(λ, fftshift(abs2.(grad_rs.initial_field_gradient)))
ax_raman.set_xlim(1400, 1700)
ax_raman.set_ylim(bottom=1)
ax_raman.set_xlabel("λ (nm)")
ax_raman.set_ylabel("Raman peak\nsensitivity (a.u)")
ax_raman.legend([raman_line], ["F = 1.045"], loc="upper left")

adj_line, = ax_filtered.semilogy(λ, fftshift(abs2.(grad_adj.initial_field_gradient)))
ad_line, = ax_filtered.semilogy(λ, fftshift(abs2.(grad_ad.initial_field_gradient)), "--")
ax_filtered.set_xlim(1530, 1675)
ax_filtered.set_ylim(bottom=1e4)
ax_filtered.set_xlabel("λ (nm)")
ax_filtered.set_ylabel("Filtered output\nsensitivity (a.u)")
ax_filtered.legend([adj_line, ad_line], ["Adjoint: F = 0.226 (no Raman), 0.271 (with)", "Autodiff "], loc="upper left")

ax_filter.plot(λ, opt_filt)
ax_filter.set_xlim(1550, 1680)
ax_filter.set_xlabel("λ (nm)")
ax_filter.set_ylabel("Transmission")

tight_layout()
subplots_adjust(wspace=0.35, hspace=0.75)
display(gcf())